In [10]:
!pip install transformers torch

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [12]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [13]:
def generate_reply(chat_history, user_message):

    # Add user message
    chat_history.append({
        "role": "user",
        "content": user_message
    })

    # Convert to model format
    prompt = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate response
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    # Decode only new tokens
    reply = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    # Save response
    chat_history.append({
        "role": "assistant",
        "content": reply
    })

    return chat_history, reply

In [14]:
def run_chatbot():

    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful AI assistant."
        }
    ]

    while True:

        user_input = input("User: ")

        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye!")
            break

        chat_history, reply = generate_reply(chat_history, user_input)

        print("Chatbot:", reply)

In [15]:
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: hello
Chatbot: Hello! How can I assist you today? Is there anything specific you would like to know or talk about?
User:  What is Artificial Intelligence?
Chatbot: Artificial intelligence (AI) refers to the simulation of human intelligence processes by machines, especially computer systems. These processes include learning, reasoning, and self-correction.

1. **Learning**: This involves the ability for machines to "learn" from data without being explicitly programmed. Machine learning algorithms analyze patterns in input data and use them to make predictions or decisions.

2. **Reasoning**: AI systems can perform logical operations, solve complex problems, and generate new ideas based on existing knowledge.

3. **Self-Correction**: AI technologies can identify errors in their own processing and adjust themselves accordingly, improving accuracy over time.

AI applications range widely across various fields such as h